In [ ]:
import logging

logging.basicConfig(
    filename="system_monitor.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True  # مهم جدًا عشان Colab / Notebook
)

logging.info("SYSTEM STARTED")
logging.info("Test log entry")
print("Logging initialized")

Logging initialized


In [ ]:
# ============================================================
# BASELINE MODEL
# ============================================================

baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_STATE
)

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", baseline_model)
])

baseline_pipeline.fit(X_train, y_train)

y_pred = baseline_pipeline.predict(X_test)
y_prob = baseline_pipeline.predict_proba(X_test)[:, 1]

baseline_accuracy = accuracy_score(y_test, y_pred)
baseline_precision = precision_score(y_test, y_pred)
baseline_recall = recall_score(y_test, y_pred)
baseline_f1 = f1_score(y_test, y_pred)
baseline_auc = roc_auc_score(y_test, y_prob)

print("Baseline Model: Logistic Regression")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall:", round(baseline_recall, 4))
print("F1 Score:", round(baseline_f1, 4))
print("ROC AUC:", round(baseline_auc, 4))

Baseline Model: Logistic Regression
Accuracy: 0.8236
Precision: 0.6743
Recall: 0.8155
F1 Score: 0.7382
ROC AUC: 0.9178


In [ ]:
# ============================================================
# MODEL COMPARISON
# ============================================================

comparison_models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=120,
        max_depth=18,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=1,
        class_weight="balanced"
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    )
}

if XGBOOST_AVAILABLE:
    comparison_models["XGBoost"] = XGBClassifier(
        n_estimators=160,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        seed=RANDOM_STATE,
        n_jobs=1
    )

results = []
trained_pipelines = {}
failed_models = {}

for name, model in comparison_models.items():

    print(f"Training {name}...")

    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    try:

        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_test)

        if hasattr(pipe.named_steps["model"], "predict_proba"):
            y_prob = pipe.predict_proba(X_test)[:, 1]
            roc_auc = roc_auc_score(y_test, y_prob)
        else:
            roc_auc = np.nan

        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        results.append({
            "Model": name,
            "Accuracy": acc,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1,
            "ROC AUC": roc_auc
        })

        trained_pipelines[name] = pipe

    except Exception as error:

        failed_models[name] = str(error)

        print(
            f"{name} failed and was skipped. "
            f"Error: {error}"
        )

if len(results) == 0:
    raise RuntimeError(
        "No comparison models trained successfully."
    )

results_df = (
    pd.DataFrame(results)
    .sort_values(
        by=["ROC AUC", "F1 Score", "Model"],
        ascending=[False, False, True]
    )
    .reset_index(drop=True)
)

results_display_df = results_df.copy()

metric_cols = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC AUC"
]

for col in metric_cols:

    results_display_df[col] = (
        results_display_df[col] * 100
    ).round(2).astype(str) + "%"

print("\nComparison Models Results:")

display(results_display_df)

Training Decision Tree...
Training Random Forest...
Training Gradient Boosting...
Training XGBoost...

Comparison Models Results:


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,XGBoost,86.02%,81.12%,70.58%,75.48%,94.01%
1,Gradient Boosting,86.1%,81.76%,70.07%,75.46%,93.98%
2,Random Forest,83.44%,66.48%,92.21%,77.26%,93.78%
3,Decision Tree,82.76%,64.26%,97.93%,77.6%,93.37%


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]

print("Best Model:", best_model_name)

Best Model: XGBoost


In [ ]:
# ============================================================
# 8.4 PLAN B BACKUP MODEL
# ============================================================

if len(results_df) > 1:
    backup_model_name = results_df.iloc[1]["Model"]
    backup_pipeline = trained_pipelines[backup_model_name]

    print("Plan B Backup Model:", backup_model_name)

else:
    backup_model_name = None
    backup_pipeline = None

    print("No Plan B model available.")

Plan B Backup Model: Gradient Boosting
